# DermaMNIST MobileNetV2 하이퍼파라미터 탐색

2025년에 사용한 학습 코드를 2026년에 다시 정리했습니다. 체크포인트 저장 방식 등을 수정했으며, 현재 코드로는 아직 다시 학습하지 않았습니다.

ImageNet으로 사전 학습한 MobileNetV2의 분류층을 피부 병변 7개 범주에 맞게 바꾸고 전체 모델을 미세조정하는 구성입니다.

정규화 값은 기존 코드의 값을 유지했습니다. 어떤 데이터 분할에서 구한 값인지는 추가로 확인할 예정입니다.


## 1. 실행 환경

필요한 라이브러리가 없다면 아래 설치 코드의 주석을 해제합니다. 실행 과정에서 데이터와 사전 학습 가중치가 다운로드될 수 있습니다. `DATA_DIR`과 `OUTPUT_ROOT`는 사용할 경로에 맞게 설정합니다.


In [ ]:
# 필요한 경우 주석을 해제해 라이브러리를 설치합니다.
# %pip install torch torchvision medmnist numpy optuna


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import json
import random
import uuid

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms as T
from medmnist import DermaMNIST

SEED = 42
DATA_DIR = Path("./data/dermamnist")
OUTPUT_ROOT = Path("./runs/dermamnist")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 7
BATCH_SIZE = 64
EPOCHS = 30
# 2025년 코드의 정규화 값입니다. 값을 구한 데이터 분할은 확인이 필요합니다.
NORMALIZE_MEAN = [0.7632, 0.5381, 0.5615]
NORMALIZE_STD = [0.1368, 0.1587, 0.1769]


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


def create_run_directory():
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    directory = OUTPUT_ROOT / f"{stamp}_{uuid.uuid4().hex[:8]}"
    directory.mkdir(parents=True, exist_ok=False)
    return directory


seed_everything(SEED)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}; PyTorch: {torch.__version__}")


## 2. 학습 및 검증 데이터


In [ ]:
train_transform = T.Compose([
    T.Resize((336, 336)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomRotation(degrees=15),
    T.ToTensor(),
    T.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])
evaluation_transform = T.Compose([
    T.Resize((336, 336)),
    T.ToTensor(),
    T.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

train_set = DermaMNIST(split="train", root=str(DATA_DIR),
                      transform=train_transform, download=True, size=224)
val_set = DermaMNIST(split="val", root=str(DATA_DIR),
                    transform=evaluation_transform, download=True, size=224)
# The test split is loaded only in the final evaluation cell.
class_counts = np.bincount(np.asarray(train_set.labels).reshape(-1), minlength=NUM_CLASSES)
class_weights = torch.tensor(1 - class_counts / class_counts.sum(), dtype=torch.float32)
print("Training class counts:", class_counts.tolist())
print(f"Train: {len(train_set)}; validation: {len(val_set)}")


## 3. 모델과 손실 함수


In [ ]:
def build_model(pretrained=True):
    # pretrained=True in the original API selected IMAGENET1K_V1.
    weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.mobilenet_v2(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_features, NUM_CLASSES))
    return model.to(DEVICE)


class FocalLoss(nn.Module):
    def __init__(self, weight, gamma=2.0):
        super().__init__()
        self.register_buffer("weight", weight)
        self.gamma = gamma

    def forward(self, logits, targets):
        cross_entropy = F.cross_entropy(logits, targets, reduction="none")
        probability = torch.exp(-cross_entropy)
        loss = (1 - probability).pow(self.gamma) * cross_entropy
        return (self.weight[targets] * loss).mean()


def mixup(images, labels, alpha=0.2):
    mixing_weight = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    permutation = torch.randperm(len(images), device=images.device)
    mixed = mixing_weight * images + (1 - mixing_weight) * images[permutation]
    return mixed, labels, labels[permutation], mixing_weight


## 4. 체크포인트 선택

체크포인트를 에포크별 파일로 저장하고, 정확도와 손실을 기준으로 사용할 파일을 선택합니다. 두 기준에 모두 포함된 체크포인트는 기존 방식대로 앙상블에 두 번 반영합니다.


In [ ]:
def rank_checkpoints(records, candidate, metric, k):
    if metric not in {"val_acc", "val_loss"} or k < 1:
        raise ValueError("Use val_acc or val_loss and k >= 1")
    candidates = [record for record in records if record["epoch"] != candidate["epoch"]]
    candidates.append(candidate)
    if metric == "val_acc":
        key = lambda record: (-record[metric], record["epoch"])
    else:
        key = lambda record: (record[metric], record["epoch"])
    return sorted(candidates, key=key)[:k]


def selected_paths(top_accuracy, top_loss):
    # An epoch selected by both criteria receives two votes, as in the originals.
    return [record["path"] for record in top_accuracy + top_loss]


## 5. 학습

Mixup 가중치를 반영한 학습 정확도는 일반적인 검증 정확도와 별도로 기록합니다. 실행마다 새 출력 폴더를 사용합니다. 체크포인트에는 평가용 가중치를 저장하며, 최적화 상태까지 포함한 완전한 학습 재개 정보는 저장하지 않습니다.


In [ ]:
def fit(params, run_dir, *, top_k_accuracy, top_k_loss, warm_restarts=False, trial=None):
    seed_everything(SEED)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = build_model()
    criterion = FocalLoss(class_weights).to(DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": model.features.parameters(), "lr": params["lr_feature"]},
        {"params": model.classifier.parameters(), "lr": params["lr_classifier"]},
    ], weight_decay=params["weight_decay"])
    if warm_restarts:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    run_dir.mkdir(parents=True, exist_ok=False)
    top_accuracy, top_loss, history = [], [], []
    config = dict(params, seed=SEED, epochs=EPOCHS, batch_size=BATCH_SIZE,
                  input_size=336, dataset_size=224,
                  normalization_mean=NORMALIZE_MEAN, normalization_std=NORMALIZE_STD,
                  class_counts=class_counts.tolist(), class_weights=class_weights.tolist(),
                  scheduler="warm_restarts" if warm_restarts else "cosine_annealing",
                  top_k_accuracy=top_k_accuracy, top_k_loss=top_k_loss,
                  torch_version=str(torch.__version__))
    (run_dir / "config.json").write_text(json.dumps(config, indent=2))

    try:
        for epoch in range(1, EPOCHS + 1):
            model.train()
            train_loss_sum = train_correct = train_count = 0
            for images, labels in train_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE).reshape(-1).long()
                optimizer.zero_grad(set_to_none=True)
                if np.random.random() < 0.5:
                    images, targets_a, targets_b, mixing_weight = mixup(images, labels)
                    logits = model(images)
                    loss = (mixing_weight * criterion(logits, targets_a)
                            + (1 - mixing_weight) * criterion(logits, targets_b))
                    predictions = logits.argmax(1)
                    correct = (mixing_weight * (predictions == targets_a).float()
                               + (1 - mixing_weight) * (predictions == targets_b).float()).sum().item()
                else:
                    logits = model(images)
                    loss = criterion(logits, labels)
                    correct = (logits.argmax(1) == labels).sum().item()
                loss.backward()
                optimizer.step()
                train_loss_sum += loss.item() * len(labels)
                train_correct += correct
                train_count += len(labels)
            scheduler.step()

            model.eval()
            val_loss_sum = val_correct = val_count = 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(DEVICE), labels.to(DEVICE).reshape(-1).long()
                    logits = model(images)
                    val_loss_sum += criterion(logits, labels).item() * len(labels)
                    val_correct += (logits.argmax(1) == labels).sum().item()
                    val_count += len(labels)
            checkpoint_path = run_dir / f"epoch_{epoch:03d}.pth"
            record = {"epoch": epoch, "val_acc": val_correct / val_count,
                      "val_loss": val_loss_sum / val_count, "path": str(checkpoint_path)}
            previous = set(selected_paths(top_accuracy, top_loss))
            top_accuracy = rank_checkpoints(top_accuracy, record, "val_acc", top_k_accuracy)
            top_loss = rank_checkpoints(top_loss, record, "val_loss", top_k_loss)
            retained = set(selected_paths(top_accuracy, top_loss))
            if str(checkpoint_path) in retained:
                torch.save(model.state_dict(), checkpoint_path)
            for old_path in previous - retained:
                Path(old_path).unlink(missing_ok=True)
            history.append(dict(record, train_loss=train_loss_sum / train_count,
                                train_mixup_accuracy=train_correct / train_count))
            selection = {"top_accuracy": top_accuracy, "top_loss": top_loss,
                         "aggregation": "mean_logits", "duplicate_epoch_votes": True}
            (run_dir / "selection.json").write_text(json.dumps(selection, indent=2))
            (run_dir / "history.json").write_text(json.dumps(history, indent=2))
            print(f"Epoch {epoch:02d} | train loss {train_loss_sum / train_count:.4f} | "
                  f"val loss {record['val_loss']:.4f} | val accuracy {record['val_acc']:.4f}")
            if trial is not None:
                trial.report(record["val_acc"], epoch)
                if trial.should_prune():
                    import optuna
                    raise optuna.TrialPruned()
        return {"best_val_acc": top_accuracy[0]["val_acc"], "run_dir": str(run_dir)}
    finally:
        del model, optimizer, scheduler, criterion
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


## 6. 하이퍼파라미터 탐색 시작

아래 실행 플래그를 바꾸면 학습을 시작할 수 있습니다. 기본 학습 코드와 Optuna 탐색 코드는 학습률 스케줄러도 달라서, 두 결과의 차이를 하이퍼파라미터 탐색만의 효과로 해석하기는 어렵습니다.


In [ ]:
import optuna

RUN_SEARCH = False
N_TRIALS = 10

if RUN_SEARCH:
    experiment_dir = create_run_directory()
    study = optuna.create_study(
        direction="maximize",
        storage=f"sqlite:///{(experiment_dir / 'optuna.db').resolve()}",
        study_name="dermamnist_mobilenetv2",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(),
    )

    def objective(trial):
        params = {
            "lr_feature": trial.suggest_float("lr_feature", 1e-5, 5e-4, log=True),
            "lr_classifier": trial.suggest_float("lr_classifier", 5e-4, 5e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-4, 5e-3, log=True),
        }
        result = fit(params, experiment_dir / f"trial_{trial.number:03d}",
                     top_k_accuracy=2, top_k_loss=2, warm_restarts=True, trial=trial)
        trial.set_user_attr("run_dir", result["run_dir"])
        return result["best_val_acc"]

    study.optimize(objective, n_trials=N_TRIALS)
    # Selection uses peak single-checkpoint validation accuracy, as in the original.
    selected_run_dir = Path(study.best_trial.user_attrs["run_dir"])
    print("Best parameters:", study.best_params)
    print("Selected run:", selected_run_dir)
else:
    print("Set RUN_SEARCH=True to start a new study. Interrupted trials are not resumed automatically.")


## 7. 최종 평가

선택한 체크포인트에서 나온 로짓을 평균해 최종 클래스를 결정합니다. 모델과 체크포인트 선택에는 검증 데이터를 사용하고, 테스트 데이터는 마지막 평가에 사용합니다. 다만 이 테스트 데이터는 과거에 이미 결과를 확인했으므로, 재실행 결과를 새로운 데이터에 대한 첫 평가로 보지는 않습니다.


In [ ]:
def evaluate_test_ensemble(run_dir):
    result_path = run_dir / "test_metrics.json"
    if result_path.exists():
        raise FileExistsError("Test metrics already exist. Read them instead of overwriting them.")
    selection = json.loads((run_dir / "selection.json").read_text())
    paths = selected_paths(selection["top_accuracy"], selection["top_loss"])
    if not paths or any(not Path(path).is_file() for path in paths):
        raise FileNotFoundError("One or more selected checkpoints are missing.")
    test_set = DermaMNIST(split="test", root=str(DATA_DIR),
                         transform=evaluation_transform, download=True, size=224)
    loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    logits_sum = None
    labels_reference = None
    # Evaluate sequentially to avoid keeping four MobileNetV2 models on the GPU.
    for path in paths:
        model = build_model(pretrained=False)
        state = torch.load(path, map_location="cpu", weights_only=True)
        model.load_state_dict(state)
        model.eval()
        outputs, targets = [], []
        with torch.no_grad():
            for images, labels in loader:
                outputs.append(model(images.to(DEVICE)).cpu())
                targets.append(labels.reshape(-1).long())
        logits = torch.cat(outputs)
        labels = torch.cat(targets)
        if labels_reference is None:
            labels_reference = labels
            logits_sum = logits
        else:
            if not torch.equal(labels, labels_reference):
                raise ValueError("Test ordering changed between checkpoint evaluations.")
            logits_sum += logits
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    mean_logits = logits_sum / len(paths)
    predictions = mean_logits.argmax(1)
    confusion = torch.bincount(labels_reference * NUM_CLASSES + predictions,
                               minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)
    confusion = confusion.numpy()
    support = confusion.sum(axis=1)
    predicted_count = confusion.sum(axis=0)
    true_positive = np.diag(confusion)
    recall = np.divide(true_positive, support, out=np.zeros(NUM_CLASSES, dtype=float), where=support > 0)
    f1 = np.divide(2 * true_positive, support + predicted_count,
                   out=np.zeros(NUM_CLASSES, dtype=float), where=(support + predicted_count) > 0)
    metrics = {"accuracy": float((predictions == labels_reference).float().mean()),
               "macro_f1": float(f1.mean()), "balanced_accuracy": float(recall[support > 0].mean()),
               "samples": len(labels_reference), "confusion_matrix": confusion.tolist(),
               "checkpoint_votes": paths, "aggregation": "mean_logits"}
    result_path.write_text(json.dumps(metrics, indent=2))
    np.savez_compressed(run_dir / "test_predictions.npz", labels=labels_reference.numpy(),
                        predictions=predictions.numpy(), mean_logits=mean_logits.numpy())
    print(json.dumps(metrics, indent=2))
    return metrics


In [ ]:
RUN_FINAL_TEST = False
# To evaluate an existing completed run in a new session, set its directory explicitly:
# selected_run_dir = Path("./runs/dermamnist/<experiment>/<baseline-or-trial>")
if RUN_FINAL_TEST:
    if "selected_run_dir" not in globals():
        raise RuntimeError("Select a completed run before evaluating the test split.")
    test_metrics = evaluate_test_ensemble(selected_run_dir)
else:
    print("Test evaluation is disabled. Freeze model choices before setting RUN_FINAL_TEST=True.")
